In [14]:
import os
import time
import threading
import random
import sqlite3
import hashlib

In [15]:
import ipywidgets as widgets
from IPython.display import display, clear_output

In [16]:
import matplotlib.pyplot as plt
import numpy as np

In [17]:
conn = sqlite3.connect(":memory:", check_same_thread=False)
cursor = conn.cursor()

In [18]:
def hash_pwd(password):
    return hashlib.sha256(password.encode()).hexdigest()

In [19]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS users (
        username TEXT PRIMARY KEY,
        password TEXT,
        role TEXT
    )
''')
conn.commit()

In [20]:
cursor.execute("INSERT INTO users VALUES (?, ?, ?)",
               ('admin', hash_pwd('admin123'), 'Admin'))
conn.commit()

In [21]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS logs (
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
        user TEXT,
        action TEXT
    )
''')
conn.commit()

In [22]:
def add_log(user, action):
    cursor.execute("INSERT INTO logs (user, action) VALUES (?, ?)", (user, action))
    conn.commit()

In [23]:
def get_logs(limit=12):
    cursor.execute("SELECT timestamp, user, action FROM logs ORDER BY timestamp DESC LIMIT ?", (limit,))
    return cursor.fetchall()

In [24]:
session = {
    "user": None,
    "role": None,
    "active": False
}


In [25]:
class Threat:
    def __init__(self, t_type, color):
        self.type = t_type
        self.color = color
        self.angle = random.uniform(0, 2 * np.pi)
        self.distance = random.uniform(80, 100)
        self.speed = random.uniform(1, 4)

    def update_position(self):
        self.distance -= self.speed

# %%
# CELL 15: Threat Module 1 - Air
class AirThreat(Threat):
    def __init__(self):
        super().__init__("Air Fighter", "red")

# %%
# CELL 16: Threat Module 2 - Naval
class NavalThreat(Threat):
    def __init__(self):
        super().__init__("Naval Vessel", "blue")
        self.speed = random.uniform(0.5, 1.5)

In [26]:
# CELL 17: Threat Module 3 - Cyber
class CyberThreat(Threat):
    def __init__(self):
        super().__init__("Cyber Node Breach", "magenta")
        self.distance = random.uniform(10, 30)

# %%
# CELL 18: Threat Module 4 - Space
class SpaceThreat(Threat):
    def __init__(self):
        super().__init__("Rogue Satellite", "white")
        self.distance = 100

# %%
# CELL 19: Threat Module 5 - Ground
class GroundThreat(Threat):
    def __init__(self):
        super().__init__("Ground Convoy", "brown")
        self.speed = random.uniform(0.2, 1)

# %%
# CELL 20: Threat Module 6 - Drone
class DroneThreat(Threat):
    def __init__(self):
        super().__init__("Drone Swarm", "orange")
        self.speed = random.uniform(3, 6)

# %%
# CELL 21: Threat Module 7 - Missile
class MissileThreat(Threat):
    def __init__(self):
        super().__init__("Ballistic Missile", "darkred")
        self.speed = random.uniform(6, 12)

# %%
# CELL 22: Threat Module 8 - EW
class EWThreat(Threat):
    def __init__(self):
        super().__init__("Electronic Warfare", "yellow")

# %%
# CELL 23: Threat Module 9 - Submarine
class SubmarineThreat(Threat):
    def __init__(self):
        super().__init__("Nuclear Submarine", "cyan")

# %%
# CELL 24: Threat Module 10 - Bio/Chem
class BioChemThreat(Threat):
    def __init__(self):
        super().__init__("Bio/Chem Hazard", "green")

# %%
# CELL 25: Active Threats List
active_threats = []

# %%
# CELL 26: Random Threat Generator Engine
def generate_threats():
    threat_classes = [AirThreat, NavalThreat, CyberThreat, SpaceThreat, GroundThreat,
                      DroneThreat, MissileThreat, EWThreat, SubmarineThreat, BioChemThreat]
    # 30% chance to spawn a new threat every cycle, max 12 threats
    if random.random() > 0.7 and len(active_threats) < 12:
        new_threat = random.choice(threat_classes)()
        active_threats.append(new_threat)
        add_log("SYSTEM", f"Detected: {new_threat.type}")

# %%
# CELL 27: Matplotlib Radar Plot Initialization
plt.ioff() # Prevent duplicate inline displays
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(5, 5))
fig.patch.set_facecolor('#001100')
ax.set_facecolor('#002200')
ax.tick_params(colors='lightgreen')
ax.grid(color='green', linestyle='--', linewidth=0.5)
plt.close(fig) # We will manually display this figure later

# %%
# CELL 28: Radar Rendering Function
def render_radar():
    ax.clear()
    ax.set_ylim(0, 100)
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels([])

    # Draw sweep line
    scan_angle = (time.time() % 4) / 4 * 2 * np.pi
    ax.plot([scan_angle, scan_angle], [0, 100], color='lightgreen', linewidth=2)

    # Plot threats
    for t in active_threats:
        t.update_position()
        ax.plot(t.angle, t.distance, marker='o', color=t.color, markersize=8)

    # Remove threats that hit the center
    active_threats[:] = [t for t in active_threats if t.distance > 0]
    display(fig)

# %%
# CELL 29: Alert Level Indicator Widget
alert_html = widgets.HTML(value="<h2 style='color:green;'>STATUS: SECURE</h2>")

def update_alert_level():
    if len(active_threats) > 7:
        alert_html.value = "<h2 style='color:red;'>STATUS: CRITICAL THREAT LEVEL</h2>"
    elif len(active_threats) > 3:
        alert_html.value = "<h2 style='color:orange;'>STATUS: ELEVATED ALERT</h2>"
    else:
        alert_html.value = "<h2 style='color:green;'>STATUS: SECURE</h2>"

# %%
# CELL 30: Command Execution Buttons
btn_intercept = widgets.Button(description="Intercept Nearest", button_style="warning")
btn_emp = widgets.Button(description="Trigger EMP (Admin)", button_style="danger")
btn_clear = widgets.Button(description="Clear Radar Data", button_style="info")

# %%
# CELL 31: Command Execution Logic
def on_intercept(b):
    if active_threats:
        # Sort by distance (closest first)
        active_threats.sort(key=lambda x: x.distance)
        destroyed = active_threats.pop(0)
        add_log(session['user'], f"Intercepted {destroyed.type}")

def on_emp(b):
    if session['role'] == 'Admin':
        active_threats.clear()
        add_log(session['user'], "EMP DEPLOYED. All local threats neutralized.")
    else:
        add_log(session['user'], "EMP FAILED: Insufficient Privileges.")

def on_clear(b):
    active_threats.clear()
    add_log(session['user'], "Radar cache cleared.")

btn_intercept.on_click(on_intercept)
btn_emp.on_click(on_emp)
btn_clear.on_click(on_clear)

# %%
# CELL 32: UI Display Output Areas
radar_out = widgets.Output()
log_out = widgets.Output(layout=widgets.Layout(width='450px', border='1px solid green', padding='10px'))

# %%
# CELL 33: Real-Time Background Update Loop
def background_task():
    while session["active"]:
        generate_threats()
        update_alert_level()

        # Update Radar
        with radar_out:
            clear_output(wait=True)
            render_radar()

        # Update Logs
        with log_out:
            clear_output(wait=True)
            print("=== SYSTEM ACTIVITY LOG ===")
            for log in get_logs(15):
                print(f"[{log[0][-8:]}] {log[1]}: {log[2]}")

        time.sleep(1) # Refresh every 1 second

# %%
# CELL 34: Dashboard Layout Assembly
controls = widgets.HBox([btn_intercept, btn_emp, btn_clear])
dashboard_ui = widgets.VBox([
    alert_html,
    controls,
    widgets.HBox([radar_out, log_out])
])

# %%
# CELL 35: Login UI Widgets
login_user = widgets.Text(description="Username:")
login_pass = widgets.Password(description="Password:")
login_btn = widgets.Button(description="Secure Login", button_style="success")
login_msg = widgets.Output()

login_ui = widgets.VBox([
    widgets.HTML("<h2>DEFENSE NETWORK SECURE LOGIN</h2>"),
    login_user, login_pass, login_btn, login_msg
])

# %%
# CELL 36: Authentication Logic
def authenticate(username, password):
    cursor.execute("SELECT role FROM users WHERE username=? AND password=?",
                   (username, hash_pwd(password)))
    result = cursor.fetchone()
    return result[0] if result else None

# %%
# CELL 37: Login Button Handler & View Switcher
main_display = widgets.Output()

def on_login(b):
    role = authenticate(login_user.value, login_pass.value)
    with login_msg:
        clear_output()
        if role:
            session["user"] = login_user.value
            session["role"] = role
            session["active"] = True
            add_log(session["user"], f"Access granted. Role: {role}")

            # Disable EMP button for Operator role
            if role != "Admin":
                btn_emp.disabled = True
                btn_emp.tooltip = "Admin privileges required"

            # Switch view to dashboard
            with main_display:
                clear_output()
                display(dashboard_ui)

            # Start radar loop in background thread
            thread = threading.Thread(target=background_task)
            thread.daemon = True # Kills thread when notebook stops
            thread.start()
        else:
            print("🚨 Access Denied: Invalid credentials.")

login_btn.on_click(on_login)

# %%
# CELL 38: Launch the Application
with main_display:
    display(login_ui)
display(main_display)

Output()